In [1]:
print("hola")

hola


In [2]:
import googlemaps
from datetime import datetime
import pandas as pd
import os
from dotenv import load_dotenv

# Carga las variables del archivo .env
load_dotenv()

# Obtiene la clave de la variable de entorno
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

# Tu ubicación actual (u origen)
origen = "Terminal de omnibus Mariano Moreno, Rosario, Argentina"

# Lista de 5 lugares (pueden ser nombres o direcciones)
lugares = ["Bar El Cairo", "Alto Rosario Shopping", "Museo Castagnino", "VIP Rosario", "Ruffo coffee co."]

nodos = [origen] + lugares

# Obtener coordenadas y horarios (Places & Geocoding)
datos_lugares = []
for p in nodos:
    res = gmaps.geocode(p)
    if res:
        place_id = res[0]['place_id']
        # Traemos detalles adicionales como el horario
        detalles = gmaps.place(place_id=place_id, fields=['name', 'opening_hours', 'geometry'])
        
        nombre = detalles['result']['name']
        coord = detalles['result']['geometry']['location']
        # Obtenemos el horario de hoy o un mensaje si no tiene
        horario = detalles['result'].get('opening_hours', {}).get('weekday_text', "No disponible")
        
        datos_lugares.append({
            'nombre': nombre,
            'coords': coord,
            'horarios': horario
        })


In [4]:
#guardar la matriz_res
#df = pd.DataFrame(matriz_res['rows'])
#df.to_csv('matriz_res.csv', index=False)

import pickle
with open('datos_lugares.pkl', 'wb') as f:
    pickle.dump(datos_lugares, f)

In [5]:
with open('datos_lugares.pkl', 'rb') as f:
    datos_lugares = pickle.load(f)

In [3]:
# Cálculo de la Matriz "Todos contra Todos"
coords_solo = [d['coords'] for d in datos_lugares]
nombres_solo = [d['nombre'] for d in datos_lugares]

matriz_res = gmaps.distance_matrix(
    origins=coords_solo,
    destinations=coords_solo,
    mode="driving", 
    departure_time=datetime.now()
)

# Estructurar la Matriz de Distancias en un DataFrame
distancias_data = []
for fila in matriz_res['rows']:
    distancias_data.append([elemento['distance']['text'] for elemento in fila['elements']])

df_matriz = pd.DataFrame(distancias_data, index=nombres_solo, columns=nombres_solo)

# Mostrar Información de Horarios y la Matriz
print("--- HORARIOS DE APERTURA ---")
for lugar in datos_lugares:
    print(f"{lugar['nombre']}:")
    # Imprimimos solo el horario de hoy para no saturar la consola
    print(f"  {lugar['horarios']}\n")

--- HORARIOS DE APERTURA ---
Rosario:
  ['Monday: Open 24 hours', 'Tuesday: Open 24 hours', 'Wednesday: Open 24 hours', 'Thursday: Open 24 hours', 'Friday: Open 24 hours', 'Saturday: Open 24 hours', 'Sunday: Open 24 hours']

El Cairo:
  ['Monday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Tuesday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Wednesday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Thursday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Friday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Saturday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Sunday: 4:00\u2009–\u200911:00\u202fPM']

Alto Rosario Shopping:
  ['Monday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Tuesday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Wednesday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Thursday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Friday: 10:00\u202fAM\u2009–\u20091:00\u202fAM', 'Saturday: 10:00\u202fAM\u2009–\u20091:00\u202fAM', 'Sunday: 10:00\u202fAM\u2009–\u200912:00\u202fAM']

Museo Municipal de Be

In [9]:
df_matriz

,Rosario,El Cairo,Alto Rosario Shopping,Museo Municipal de Bellas Artes Juan B. Castagnino,VIP Rosario,Ruffo Coffee Co.
Rosario,1 m,4.1 km,3.0 km,3.5 km,5.8 km,2.4 km
El Cairo,3.2 km,1 m,5.4 km,2.9 km,2.3 km,3.3 km
Alto Rosario Shopping,2.4 km,4.6 km,1 m,4.9 km,5.2 km,2.6 km
Museo Municipal de Bellas Artes Juan B. Castagnino,3.0 km,3.4 km,4.6 km,1 m,3.8 km,2.5 km
VIP Rosario,4.9 km,1.7 km,6.1 km,3.5 km,1 m,4.5 km
Ruffo Coffee Co.,2.0 km,3.3 km,2.1 km,2.6 km,3.9 km,1 m


In [10]:
# Guardar df_matriz en un archivo CSV
df_matriz.to_csv('matriz_distancias.csv')
